# Semantic Caching for AI Agents with LLM Verification

This guide demonstrates how to implement semantic caching for complex agentic workflows where embedding similarity alone isn't sufficient. When queries trigger tool calls, RAG retrieval, or multi-step reasoning, you need stricter cache validation to avoid returning incorrect cached responses.

## The Problem

Simple semantic caching works well for straightforward Q&A:
- "What is the capital of France?" ≈ "Capital of France?" → Same answer ✓

But for complex agentic queries, high embedding similarity doesn't guarantee the same answer:
- "Show Q3 2024 revenue by region" ≈ "Show Q3 2023 revenue by region" → **Different answers!**
- "List my open support tickets" ≈ "List my closed support tickets" → **Different tool calls!**

## The Solution: LLM-Verified Caching

We use a two-step approach:
1. **Vector search** finds semantically similar candidates (fast, cheap)
2. **LLM judge** verifies if candidates are truly equivalent (accurate, selective)

The key insight: **only send the cached questions to the LLM judge, not the full responses**. This keeps context small and costs low.

## Prerequisites

- RedisVL installed: `pip install redisvl`
- Running Redis instance ([Redis 8+](https://redis.io/downloads/) or [Redis Cloud](https://redis.io/cloud))
- OpenAI API key

In [ ]:
import os
import getpass
import time
import json
from typing import Optional, List, Dict, Any

from openai import OpenAI

os.environ["TOKENIZERS_PARALLELISM"] = "False"

api_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

## Initialize the Semantic Cache

We'll use `SemanticCache` with custom filterable fields to store metadata about each cached query (intent, entities, required tools).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from redisvl.extensions.cache.llm import SemanticCache
from redisvl.utils.vectorize import OpenAITextVectorizer

# Initialize cache with filterable fields for structured matching
# IMPORTANT: Set ttl=None to prevent automatic TTL refresh on check()
cache = SemanticCache(
    name="agentic_cache",
    redis_url="redis://localhost:6379",
    distance_threshold=0.3,  # Wider threshold - we'll use LLM to verify
    ttl=None,  # Explicitly disable TTL - entries persist until manually deleted
    vectorizer=OpenAITextVectorizer(api_config={"api_key": api_key}),
    filterable_fields=[
        {"name": "intent", "type": "tag"},
        {"name": "tools_required", "type": "tag"},
    ],
    overwrite=True,
)

# Verify TTL is disabled
print(f"Cache TTL setting: {cache.ttl}")

## Seed the Cache with Sample Data

Let's populate the cache with realistic agentic queries. These represent cached responses from previous tool calls and RAG retrievals.

In [ ]:
# Sample cache entries representing complex agentic queries
cache_entries = [
    {
        "prompt": "What was our Q3 2024 revenue broken down by region?",
        "response": "Q3 2024 Revenue by Region:\n- North America: $4.2M\n- Europe: $2.8M\n- Asia Pacific: $1.9M\n- Total: $8.9M",
        "filters": {"intent": "revenue_query", "tools_required": "database_query"},
        "metadata": {"time_period": "Q3-2024", "data_type": "financial"}
    },
    {
        "prompt": "What was our Q3 2023 revenue broken down by region?",
        "response": "Q3 2023 Revenue by Region:\n- North America: $3.8M\n- Europe: $2.4M\n- Asia Pacific: $1.5M\n- Total: $7.7M",
        "filters": {"intent": "revenue_query", "tools_required": "database_query"},
        "metadata": {"time_period": "Q3-2023", "data_type": "financial"}
    },
    {
        "prompt": "List all open support tickets assigned to me",
        "response": "Open tickets:\n1. TICK-1234: Login issues (High)\n2. TICK-1237: API timeout (Medium)\n3. TICK-1242: Dashboard not loading (Low)",
        "filters": {"intent": "ticket_query", "tools_required": "ticketing_api"},
        "metadata": {"ticket_status": "open"}
    },
    {
        "prompt": "List all closed support tickets assigned to me",
        "response": "Closed tickets (last 30 days):\n1. TICK-1201: Password reset (Resolved)\n2. TICK-1215: Data export (Resolved)\n3. TICK-1228: Permission error (Resolved)",
        "filters": {"intent": "ticket_query", "tools_required": "ticketing_api"},
        "metadata": {"ticket_status": "closed"}
    },
    {
        "prompt": "What are the current inventory levels for SKU-A100?",
        "response": "SKU-A100 Inventory:\n- Warehouse East: 1,250 units\n- Warehouse West: 890 units\n- In Transit: 200 units\n- Total Available: 2,340 units",
        "filters": {"intent": "inventory_query", "tools_required": "inventory_api"},
        "metadata": {"sku": "A100"}
    },
    {
        "prompt": "What are the current inventory levels for SKU-B200?",
        "response": "SKU-B200 Inventory:\n- Warehouse East: 450 units\n- Warehouse West: 320 units\n- In Transit: 50 units\n- Total Available: 820 units",
        "filters": {"intent": "inventory_query", "tools_required": "inventory_api"},
        "metadata": {"sku": "B200"}
    },
    {
        "prompt": "Summarize the key points from our Q3 board meeting",
        "response": "Q3 Board Meeting Summary:\n1. Revenue exceeded targets by 12%\n2. New product launch approved for Q1 2025\n3. Hiring freeze lifted for engineering\n4. International expansion to proceed",
        "filters": {"intent": "document_summary", "tools_required": "rag_retrieval"},
        "metadata": {"document_type": "meeting_notes", "quarter": "Q3"}
    },
    {
        "prompt": "What is our company's refund policy?",
        "response": "Refund Policy:\n- Full refund within 30 days of purchase\n- 50% refund between 30-60 days\n- No refunds after 60 days\n- Digital products: 14-day refund window",
        "filters": {"intent": "policy_query", "tools_required": "rag_retrieval"},
        "metadata": {"policy_type": "refund"}
    },
    {
        "prompt": "How many active users do we have this month?",
        "response": "Active Users (Current Month):\n- Daily Active: 45,230\n- Weekly Active: 128,450\n- Monthly Active: 312,800\n- Growth vs Last Month: +8.3%",
        "filters": {"intent": "metrics_query", "tools_required": "analytics_api"},
        "metadata": {"metric_type": "user_activity", "time_scope": "current_month"}
    },
    {
        "prompt": "How many active users did we have last month?",
        "response": "Active Users (Last Month):\n- Daily Active: 41,200\n- Weekly Active: 118,900\n- Monthly Active: 288,750\n- Growth vs Prior Month: +5.1%",
        "filters": {"intent": "metrics_query", "tools_required": "analytics_api"},
        "metadata": {"metric_type": "user_activity", "time_scope": "last_month"}
    },
]

# Store all entries
print("Seeding cache with sample data...")
for entry in cache_entries:
    cache.store(
        prompt=entry["prompt"],
        response=entry["response"],
        filters=entry["filters"],
        metadata=entry["metadata"]
    )
print(f"Stored {len(cache_entries)} cache entries")

## Entry IDs and Keys

Each cache entry has two identifiers that are useful for direct access:

- **`entry_id`**: A deterministic hash of the prompt + filters. Used to identify the entry within the cache.
- **`key`**: The full Redis key (prefix + entry_id). Used for direct Redis operations.

This is important for the "fetch questions only" architecture - we retrieve entry_ids with the candidates, then fetch only the winning response.

In [ ]:
# Store an entry and capture the returned key
test_key = cache.store(
    prompt="What is the status of Project Alpha?",
    response="Project Alpha is 75% complete, on track for Q2 delivery.",
    filters={"intent": "project_status", "tools_required": "project_api"},
    metadata={"project_id": "alpha"}
)
print(f"Full Redis key: {test_key}")

# Check and see both entry_id and key in the response
result = cache.check(
    prompt="What is the status of Project Alpha?",
    return_fields=["entry_id", "prompt", "response"]
)
print(f"Entry ID: {result[0]['entry_id']}")
print(f"Key: {result[0]['key']}")

# Fetch directly by entry_id (this is what we do after LLM verification)
fetched = cache._index.fetch(result[0]['entry_id'])
print(f"Fetched response: {fetched['response'][:50]}...")

## Advanced Metadata Filtering

For agentic workflows, you can combine semantic search with metadata filters to narrow down candidates before LLM verification. This is especially useful for multi-tenant systems or when queries have known constraints.

In [ ]:
from redisvl.query.filter import Tag

# Filter by intent - only search within revenue queries
intent_filter = Tag("intent") == "revenue_query"

results = cache.check(
    prompt="Show me the quarterly revenue",
    filter_expression=intent_filter,
    num_results=5,
    return_fields=["prompt", "intent", "vector_distance"]
)

print("=== Filtered by intent='revenue_query' ===")
for r in results:
    print(f"  {r['prompt'][:50]}... (distance={r['vector_distance']:.3f})")

In [ ]:
# Filter by tools_required - useful when you know which tool the query needs
tools_filter = Tag("tools_required") == "ticketing_api"

results = cache.check(
    prompt="Show me my tickets",
    filter_expression=tools_filter,
    num_results=5,
    return_fields=["prompt", "tools_required", "vector_distance"]
)

print("\n=== Filtered by tools_required='ticketing_api' ===")
for r in results:
    print(f"  {r['prompt'][:50]}... (distance={r['vector_distance']:.3f})")

## TTL Behavior Note

**Important**: When `ttl` is set on the cache, the `check()` method automatically refreshes TTL on all matched entries. This can unexpectedly extend the lifetime of entries.

For agentic caching where you want explicit control over entry lifecycle, we recommend:
- Set `ttl=None` (as we did above) for persistent entries
- Manually delete stale entries when underlying data changes
- Use metadata to track entry freshness if needed

In [ ]:
# Verify TTL is still None after check operations
print(f"Cache TTL after operations: {cache.ttl}")

# If you need TTL, set it explicitly per-entry at store time:
# cache.store(prompt="...", response="...", ttl=3600)  # 1 hour TTL for this entry only

## The Problem: False Positives with Embedding Similarity

Let's demonstrate why embedding similarity alone fails for complex queries. These queries are semantically similar but require **different** answers.

In [ ]:
# Test queries that are similar but NOT equivalent
test_queries = [
    "Show me Q3 2024 revenue by region",      # Should match Q3 2024, NOT Q3 2023
    "What are my open tickets?",               # Should match open, NOT closed
    "Current inventory for SKU-A100",          # Should match A100, NOT B200
    "How many active users this month?",       # Should match current, NOT last month
]

print("=== Without LLM Verification (Embedding Only) ===\n")
for query in test_queries:
    results = cache.check(prompt=query, num_results=3, return_fields=["prompt", "response", "vector_distance"])
    print(f"Query: {query}")
    print(f"Top match (distance={results[0]['vector_distance']:.3f}): {results[0]['prompt'][:60]}...")
    if len(results) > 1:
        print(f"2nd match (distance={results[1]['vector_distance']:.3f}): {results[1]['prompt'][:60]}...")
    print()

Notice how the embedding distances between correct and incorrect matches are often very close. A simple threshold can't reliably distinguish "Q3 2024" from "Q3 2023" or "open tickets" from "closed tickets".

## Implement the LLM Judge

The LLM judge determines if a candidate cached question is semantically equivalent to the user's query. Key design decisions:

1. **Only send questions, not responses** - keeps context small
2. **Ask for structured output** - easier to parse
3. **Include reasoning** - helps with debugging and confidence

In [ ]:
def llm_judge(
    user_query: str,
    candidate_prompts: List[str],
    model: str = "gpt-4o-mini"
) -> Optional[int]:
    """
    Use LLM to determine which candidate (if any) is semantically equivalent.

    Args:
        user_query: The user's current query
        candidate_prompts: List of cached question prompts to evaluate
        model: OpenAI model to use for judgment

    Returns:
        Index of the matching candidate, or None if no match
    """
    if not candidate_prompts:
        return None

    candidates_text = "\n".join(
        f"{i}. {prompt}" for i, prompt in enumerate(candidate_prompts)
    )

    prompt = f"""You are evaluating whether cached questions can answer a user's query.

USER QUERY: {user_query}

CACHED QUESTIONS:
{candidates_text}

Determine if any cached question would produce the EXACT SAME answer as the user query.
Consider: time periods, specific IDs/SKUs, status filters, and any other distinguishing details.

Two questions are equivalent ONLY if:
- They ask for the same information
- They would require the same tool calls / data retrieval
- The answers would be interchangeable

Respond with JSON only:
{{"match_index": <index or null>, "reasoning": "<brief explanation>"}}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=150
    )

    result = json.loads(response.choices[0].message.content)
    return result.get("match_index")

## Option 1: Fetch Questions Only Architecture

This is the recommended pattern. We fetch only the question prompts and entry IDs (not the full responses), send questions to the LLM judge, then fetch only the winning response.

In [ ]:
def check_cache_with_verification(
    query: str,
    num_candidates: int = 3,
    high_confidence_threshold: float = 0.05,
) -> Dict[str, Any]:
    """
    Check cache with LLM verification for complex queries.

    Architecture:
    1. Vector search for candidates (return prompts + entry_ids, NOT responses)
    2. If very high confidence, skip LLM verification
    3. Otherwise, LLM judge evaluates candidates
    4. Fetch only the winning response

    Returns dict with: hit, response, entry_id, distance, verified, timing
    """
    start_time = time.time()
    result = {
        "hit": False,
        "response": None,
        "entry_id": None,
        "distance": None,
        "verified": False,
        "llm_called": False,
        "timing_ms": 0
    }

    # Step 1: Get candidates (prompts only, NOT responses)
    candidates = cache.check(
        prompt=query,
        num_results=num_candidates,
        return_fields=["prompt", "entry_id", "vector_distance"]  # No response!
    )

    if not candidates:
        result["timing_ms"] = (time.time() - start_time) * 1000
        return result

    best = candidates[0]
    distance = float(best.get("vector_distance", 1.0))

    # Step 2: High confidence - skip verification
    if distance < high_confidence_threshold:
        # Fetch the response for this entry
        entry = cache._index.fetch(best["entry_id"])
        result.update({
            "hit": True,
            "response": entry.get("response") if entry else None,
            "entry_id": best["entry_id"],
            "distance": distance,
            "verified": False,  # Skipped verification due to high confidence
        })
        result["timing_ms"] = (time.time() - start_time) * 1000
        return result

    # Step 3: LLM verification
    result["llm_called"] = True
    candidate_prompts = [c["prompt"] for c in candidates]
    match_idx = llm_judge(query, candidate_prompts)

    if match_idx is not None and 0 <= match_idx < len(candidates):
        matched = candidates[match_idx]
        # Step 4: Fetch ONLY the winning response
        entry = cache._index.fetch(matched["entry_id"])
        result.update({
            "hit": True,
            "response": entry.get("response") if entry else None,
            "entry_id": matched["entry_id"],
            "distance": float(matched.get("vector_distance", 0)),
            "verified": True,
        })

    result["timing_ms"] = (time.time() - start_time) * 1000
    return result


## Demonstrate LLM-Verified Caching

Now let's test the same queries with LLM verification and see how it correctly identifies matches.

In [ ]:
print("=== With LLM Verification ===\n")
for query in test_queries:
    result = check_cache_with_verification(query)
    print(f"Query: {query}")
    print(f"  Hit: {result['hit']}, LLM Called: {result['llm_called']}, Time: {result['timing_ms']:.0f}ms")
    if result['hit']:
        # Show first line of response
        first_line = result['response'].split('\n')[0] if result['response'] else "N/A"
        print(f"  Response: {first_line}")
    print()

## Compare: With vs Without Verification

Let's run a comprehensive comparison showing accuracy and performance differences.

In [ ]:
# Extended test set including edge cases
comparison_queries = [
    # Should find exact/near-exact matches
    ("What is our company's refund policy?", True, "refund"),
    ("Tell me about the refund policy", True, "refund"),

    # Should correctly distinguish time periods
    ("Q3 2024 revenue breakdown", True, "Q3 2024"),
    ("Q3 2023 revenue breakdown", True, "Q3 2023"),

    # Should correctly distinguish status
    ("Show my open support tickets", True, "open"),
    ("Show my closed support tickets", True, "closed"),

    # Should be cache misses (no matching entry)
    ("What is our vacation policy?", False, None),
    ("Show me Q1 2024 revenue", False, None),
]

print("=== Accuracy Comparison ===\n")
print(f"{'Query':<45} {'Expected':<10} {'No Verify':<12} {'Verified':<10}")
print("-" * 80)

correct_without = 0
correct_with = 0

for query, should_hit, expected_keyword in comparison_queries:
    # Without verification (just use top result if within threshold)
    results_no_verify = cache.check(prompt=query, num_results=1)
    hit_no_verify = len(results_no_verify) > 0
    correct_no_verify = False
    if should_hit and hit_no_verify and expected_keyword:
        correct_no_verify = expected_keyword.lower() in results_no_verify[0].get('response', '').lower()
    elif not should_hit and not hit_no_verify:
        correct_no_verify = True

    # With verification
    result_verified = check_cache_with_verification(query)
    correct_verified = False
    if should_hit and result_verified['hit'] and expected_keyword:
        correct_verified = expected_keyword.lower() in (result_verified.get('response') or '').lower()
    elif not should_hit and not result_verified['hit']:
        correct_verified = True

    correct_without += correct_no_verify
    correct_with += correct_verified

    status_no = "✓" if correct_no_verify else "✗"
    status_ver = "✓" if correct_verified else "✗"

    print(f"{query[:43]:<45} {'Hit' if should_hit else 'Miss':<10} {status_no:<12} {status_ver:<10}")

print("-" * 80)
print(f"{'Accuracy:':<45} {'':<10} {correct_without}/{len(comparison_queries):<12} {correct_with}/{len(comparison_queries):<10}")

## Performance Considerations

The LLM verification adds latency but provides accuracy. Here's how to optimize:

| Scenario | Strategy |
|----------|----------|
| Very high confidence (distance < 0.05) | Skip LLM verification |
| Moderate confidence (0.05 - 0.25) | Use LLM verification |
| Low confidence (> 0.25) | Treat as cache miss |

You can also use a smaller/faster model for verification (e.g., `gpt-4o-mini`) since it only needs to compare questions, not generate responses.

In [ ]:
# Measure timing breakdown
print("=== Timing Breakdown ===\n")

# Cache-only timing
start = time.time()
for _ in range(10):
    cache.check(prompt="What is our refund policy?", num_results=3)
cache_only_ms = (time.time() - start) / 10 * 1000

# With verification timing
start = time.time()
for _ in range(3):
    check_cache_with_verification("What is our refund policy?")
with_verify_ms = (time.time() - start) / 3 * 1000

print(f"Cache lookup only (avg): {cache_only_ms:.1f}ms")
print(f"With LLM verification (avg): {with_verify_ms:.1f}ms")
print(f"LLM overhead: ~{with_verify_ms - cache_only_ms:.0f}ms")
print(f"\nNote: LLM verification is still faster than a full LLM generation + tool execution pipeline")

## Key Takeaways

1. **Embedding similarity alone fails for complex queries** - Time periods, IDs, and status filters can't be reliably distinguished by embeddings

2. **LLM verification adds accuracy without bloating context** - Only send cached questions to the judge, not full responses

3. **Use tiered confidence** - Skip verification for very high confidence matches, use it for borderline cases

4. **Never delete unselected candidates** - They're valid cache entries for other queries

5. **The overhead is worth it** - LLM verification (~200-500ms) is still much faster than full tool execution pipelines (seconds to minutes)

## Cleanup

In [ ]:
cache.delete()
print("Cache deleted successfully")